# Normalización y cálculo de AM

In [1]:
import pandas as pd
import numpy as np

def procesar_y_crear_nuevo_csv():
    PESOS = {
        "normalized_impact": 0.13, "excel_lider": 0.08, "output": 0.08, "lider": 0.05,
        "not_own_journals_output": 0.03, "own_journals": 0.03, "excel": 0.02, "q1": 0.02,
        "colab": 0.02, "open_access": 0.02, "stp": 0.02, "ik": 0.10, "patents": 0.10,
        "tech_impact": 0.10, "plumx": 0.03, "web_size": 0.03, "sdg": 0.05,
        "female_stp": 0.03, "overton": 0.03
    }

    try:
        df = pd.read_csv('instituciones_chilenas_limpias.csv')

        # Calcular AM
        if 'plumx' in df.columns and 'mendeley' in df.columns:
            df['AM'] = df['plumx'] * 0.7 + df['mendeley'] * 0.3
        elif 'plumx' in df.columns:
            df['AM'] = df['plumx']
        else:
            df['AM'] = df.get('mendeley', 0)

        # Eliminar columnas
        df = df.drop(columns=['mendeley', 'plumx'], errors='ignore')

        # Actualizar pesos
        PESOS_ACTUALIZADOS = PESOS.copy()
        if 'plumx' in PESOS_ACTUALIZADOS:
            PESOS_ACTUALIZADOS['AM'] = PESOS_ACTUALIZADOS.pop('plumx')

        # Convertir y limpiar datos
        for columna in PESOS_ACTUALIZADOS:
            if columna in df.columns:
                df[columna] = pd.to_numeric(df[columna], errors='coerce').fillna(0)

        # Filtrar por año
        if 'years' in df.columns:
            año = 2023 if 2023 in df['years'].unique() else df['years'].max()
            df = df[df['years'] == año].copy()

        # Normalización
#        for columna in PESOS_ACTUALIZADOS:
#            if columna in df.columns:
#                min_val, max_val = df[columna].min(), df[columna].max()
#                if max_val > min_val:
#                    df[columna] = (df[columna] - min_val) / (max_val - min_val) * 100
#                else:
#                    df[columna] = 0

        # Calcular puntuación
        df['PuntajeOriginalNormalizado'] = sum(df[col] * peso for col, peso in PESOS_ACTUALIZADOS.items() if col in df.columns)

        # Normalizar puntuación final
#        min_punt, max_punt = df['PuntajeOriginalNormalizado'].min(), df['PuntajeOriginalNormalizado'].max()
#        if max_punt > min_punt:
#            df['PuntajeOriginalNormalizado'] = (df['PuntajeOriginalNormalizado'] - min_punt) / (max_punt - min_punt) * 100

        # Calcular ranking
        if 'Institucion' in df.columns:
            ranking_df = df.groupby('Institucion')['PuntajeOriginalNormalizado'].mean().reset_index()
            ranking_df = ranking_df.sort_values('PuntajeOriginalNormalizado', ascending=False)
            ranking_df['Ranking'] = range(1, len(ranking_df) + 1)
            df['Ranking'] = df['Institucion'].map(ranking_df.set_index('Institucion')['Ranking'])

        # Guardar
        df.to_csv("instituciones_chilenas_LIMPIAS_con_ranking_y_AM.csv", index=False, encoding='utf-8')

        print("✅ Proceso completado con normalización")
        print(f"📊 Rango de puntuaciones: {df['PuntajeOriginalNormalizado'].min():.2f} - {df['PuntajeOriginalNormalizado'].max():.2f}")
        return True

    except FileNotFoundError:
        print("❌ Error: Archivo 'instituciones_chilenas_limpias.csv' no encontrado.")
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

if __name__ == "__main__":
    if procesar_y_crear_nuevo_csv():
        print("✅ Archivo creado con éxito con datos normalizados")

✅ Proceso completado con normalización
📊 Rango de puntuaciones: 0.16 - 3983.17
✅ Archivo creado con éxito con datos normalizados


# Pesos

In [2]:
# Cargar el archivo CSV
df = pd.read_csv('instituciones_chilenas_LIMPIAS_con_ranking_y_AM.csv')

PESOS = {
    "normalized_impact": 0.13, "excel_lider": 0.08, "output": 0.08, "lider": 0.05,
    "not_own_journals_output": 0.03, "own_journals": 0.03, "excel": 0.02, "q1": 0.02,
    "colab": 0.02, "open_access": 0.02, "stp": 0.02, "ik": 0.10, "patents": 0.10,
    "tech_impact": 0.10, "AM": 0.03, "web_size": 0.03, "sdg": 0.05,
    "female_stp": 0.03, "overton": 0.03
}

df

,ID,Institucion,years,idp,lang,excel_lider,normalized_impact,normalized_impact_leadership,output,stp,...,patents,open_access,not_own_journals_output,own_journals,overton,sdg,female_stp,AM,PuntajeOriginalNormalizado,Ranking
0,1150,Pontificia Universidad Catolica de Chile,2019-2023,1150,all,801,1.20949,0.82179,16831,10828,...,115,65.587309,16451,22,606,5901,4090,6927.8,3661.037980,2
1,1151,Pontificia Universidad Catolica de Valparaiso,2019-2023,1151,all,260,0.91052,0.76834,5052,2539,...,13,67.794933,4945,6,93,1538,967,2084.4,1045.716266,7
2,1152,Universidad Academia de Humanismo Cristiano,2019-2023,1152,all,4,0.47746,0.37079,220,148,...,0,74.545455,220,0,3,84,61,80.5,47.117979,43
3,1153,Universidad Adolfo Ibanez,2019-2023,1153,all,142,1.08523,0.84363,2649,947,...,8,63.231408,2601,3,102,755,296,1064.3,541.310708,16
4,1154,Universidad Adventista de Chile,2019-2023,1154,all,10,0.97918,0.89421,287,195,...,0,75.958188,287,0,3,84,83,110.2,58.812457,42
5,1155,Universidad Alberto Hurtado,2019-2023,1155,all,45,0.73447,0.55710,1112,638,...,0,68.255396,1107,1,37,421,268,425.9,234.047589,30
6,1156,Universidad Arturo Prat,2019-2023,1156,all,26,0.76796,0.53776,816,432,...,0,72.426471,816,0,9,275,138,344.2,167.776364,36
7,1157,Universidad Austral de Chile,2019-2023,1157,all,172,0.94005,0.72837,4999,2886,...,17,66.193239,4871,7,137,1874,1112,2206.6,1067.172071,6
8,1158,Universidad Autonoma de Chile,2019-2023,1158,all,121,1.11963,0.79087,3577,1074,...,8,69.136148,3577,0,101,1247,433,1753.7,721.478275,10
9,1159,Universidad Bernardo O'Higgins,2019-2023,1159,all,39,0.79748,0.58391,1392,543,...,23,71.479885,1392,0,11,399,205,577.7,275.644270,27


# Ranking innovación

In [3]:
# Leer y filtrar datos
df = pd.read_csv('instituciones_chilenas_limpias.csv')
df = df[df["years"] == "2019-2023"].copy()

# Columnas para impacto tecnológico
COLUMNAS_IMPACTO = ['Institucion', 'ik', 'patents', 'tech_impact']
df_impacto = df[COLUMNAS_IMPACTO].copy()

# Calcular score y ranking
df_impacto["impacto_score"] = df_impacto[['ik', 'patents', 'tech_impact']].sum(axis=1)
df_impacto["ranking"] = df_impacto["impacto_score"].rank(ascending=False, method="min")

# Mostrar resultados ordenados
df_ranking_impacto = df_impacto.sort_values("ranking")
print("--- Ranking de Instituciones por Impacto Tecnológico ---")
display(df_ranking_impacto)

# Función para simular cambios
def simular_cambio_impacto(universidad, atributo, nuevo_valor):
    df_sim = df_impacto.copy()

    mask = df_sim["Institucion"].str.contains(universidad, case=False, na=False)
    if not mask.any():
        print(f"❌ No se encontró '{universidad}'")
        return

    df_sim.loc[mask, atributo] = nuevo_valor
    df_sim["impacto_score"] = df_sim[['ik', 'patents', 'tech_impact']].sum(axis=1)
    df_sim["ranking"] = df_sim["impacto_score"].rank(ascending=False, method="min")

    print(f"🔄 Simulación: {atributo} = {nuevo_valor} para {universidad}")
    display(df_sim.sort_values("ranking"))

--- Ranking de Instituciones por Impacto Tecnológico ---


,Institucion,ik,patents,tech_impact,impacto_score,ranking
5,Pontificia Universidad Catolica de Chile,176,115,1.15,292.15,1.0
131,Universidad de Chile,203,50,1.19,254.19,2.0
143,Universidad de Concepcion,85,89,1.04,175.04,3.0
327,Universidad Tecnica Federico Santa Maria,54,82,1.39,137.39,4.0
191,Universidad de Santiago de Chile,67,52,1.28,120.28,5.0
53,Universidad Autonoma de Chile,53,8,1.59,62.59,6.0
303,"Universidad Andres Bello, Chile",52,7,0.88,59.88,7.0
149,Universidad de la Frontera,35,20,0.90,55.90,8.0
11,Pontificia Universidad Catolica de Valparaiso,42,13,0.90,55.90,8.0
197,Universidad de Talca,40,9,1.35,50.35,10.0


# Ranking Research

In [4]:
# Diccionario de pesos
PESOS_IMPACTO = {
    "normalized_impact": 0.13, "excel_lider": 0.08, "output": 0.08, "lider": 0.05,
    "not_own_journals_output": 0.03, "own_journals": 0.03, "excel": 0.02, "q1": 0.02,
    "colab": 0.02, "open_access": 0.02, "stp": 0.02
}

# Asegurar columnas existan
for col in PESOS_IMPACTO:
    if col not in df.columns:
        df[col] = 0
        print(f"⚠️ Columna '{col}' creada con 0")

# Crear DataFrame y calcular score
COLUMNAS_IMPACTO = ["Institucion"] + list(PESOS_IMPACTO.keys())
df_impacto = df[COLUMNAS_IMPACTO].copy()
df_impacto["impacto_score"] = sum(df_impacto[col] * peso for col, peso in PESOS_IMPACTO.items())
df_impacto["ranking"] = df_impacto["impacto_score"].rank(ascending=False, method="min")

# Mostrar resultados
df_ranking = df_impacto.sort_values("ranking")
print("--- Ranking de Instituciones (Score Ponderado) ---")
display(df_ranking)

# Función para simular cambios
def simular_cambio_ponderado(universidad, atributo, nuevo_valor):
    df_sim = df_impacto.copy()

    mask = df_sim["Institucion"].str.contains(universidad, case=False, na=False)
    if not mask.any():
        print(f"❌ No se encontró '{universidad}'")
        return

    df_sim.loc[mask, atributo] = nuevo_valor
    df_sim["impacto_score"] = sum(df_sim[col] * peso for col, peso in PESOS_IMPACTO.items())
    df_sim["ranking"] = df_sim["impacto_score"].rank(ascending=False, method="min")

    print(f"🔄 Simulación: {atributo} = {nuevo_valor} para {universidad}")
    display(df_sim.sort_values("ranking"))

--- Ranking de Instituciones (Score Ponderado) ---


,Institucion,normalized_impact,excel_lider,output,lider,not_own_journals_output,own_journals,excel,q1,colab,open_access,stp,impacto_score,ranking
131,Universidad de Chile,1.08933,819,18482,9215,18199,16,2416,9972,10666,65.550265,12806,3269.932618,1.0
5,Pontificia Universidad Catolica de Chile,1.20949,801,16831,8564,16451,22,2399,9532,9923,65.587309,10828,2988.058980,2.0
143,Universidad de Concepcion,1.10470,326,8715,4179,8559,9,1078,4903,5271,62.845668,5430,1524.310524,3.0
303,"Universidad Andres Bello, Chile",1.08571,188,6310,2333,6293,1,808,3181,3946,67.511886,2913,1043.761380,4.0
191,Universidad de Santiago de Chile,0.90826,260,5668,2942,5668,0,649,2840,3148,61.132675,3200,989.460727,5.0
11,Pontificia Universidad Catolica de Valparaiso,0.91052,260,5052,2535,4945,6,616,2367,2837,67.794933,2539,868.894266,6.0
47,Universidad Austral de Chile,0.94005,172,4999,2425,4871,7,545,2669,3006,66.193239,2886,864.836071,7.0
149,Universidad de la Frontera,1.05995,180,4205,2194,3925,2,481,1864,2541,68.038050,2340,724.328554,8.0
327,Universidad Tecnica Federico Santa Maria,1.13835,158,4058,1731,4058,0,614,2164,2823,61.458847,1638,691.727162,9.0
53,Universidad Autonoma de Chile,1.11963,121,3577,1200,3577,0,511,1827,2566,69.136148,1074,584.238275,10.0


# Elimina Asteriscos

In [5]:
import pandas as pd

# Leer y limpiar datos
df_rank_csv = pd.read_csv("ScimagoIR 2024 - Overall Rank - Universities - CHL.csv", sep=";", encoding="utf-8", engine="python")
df_rank_csv["Institution"] = df_rank_csv["Institution"].str.replace("*", "", regex=False).str.strip()

# Guardar archivo limpio
df_rank_csv.to_csv("ScimagoIR_2024_CHL_sin_asteriscos.csv", index=False, sep=";")
print("✅ Archivo limpio guardado como 'ScimagoIR_2024_CHL_sin_asteriscos.csv'")

✅ Archivo limpio guardado como 'ScimagoIR_2024_CHL_sin_asteriscos.csv'


# Comparación KAI (IBER) Rank - SIR Rank

In [6]:
import pandas as pd

# Función específica para agregar RankingKAI
def agregar_rankingkai_al_csv():
    """Agrega columna RankingKAI al CSV si no existe"""
    try:
        df = pd.read_csv("instituciones_chilenas_LIMPIAS_con_ranking_y_AM.csv")
        
        if "RankingKAI" not in df.columns:
            print("➕ Agregando columna 'RankingKAI'...")
            
            # Definir métricas para cálculo
            excluir = ["Institucion", "ID", "idp", "years", "lang"]
            metricas = [col for col in df.columns 
                       if pd.api.types.is_numeric_dtype(df[col]) and col not in excluir]
            
            # Calcular score si no existe
            if "impacto_score" not in df.columns:
                df["impacto_score"] = df[metricas].sum(axis=1)
            
            # Calcular ranking (menor número = mejor)
            df["RankingKAI"] = df["impacto_score"].rank(ascending=False, method="min").astype(int)
            df = df.sort_values("RankingKAI", ascending=True)
            
            # Guardar
            df.to_csv("instituciones_chilenas_LIMPIAS_con_ranking_y_AM.csv", index=False)
            print(f"✅ RankingKAI agregado. Total: {len(df)} instituciones")
            
        return df
    
    except FileNotFoundError:
        print("❌ Archivo no encontrado")
        return pd.DataFrame()

# Ejecutar
df_con_rankingkai = agregar_rankingkai_al_csv()

➕ Agregando columna 'RankingKAI'...
✅ RankingKAI agregado. Total: 57 instituciones


# Comparar universidades - Análisis avanzado (experimental)

In [ ]:
import pandas as pd
import numpy as np

class AnalizadorInstitucionesAvanzado:
    def __init__(self, archivo="instituciones_chilenas_LIMPIAS_con_ranking_y_AM.csv", years_objetivo="2019-2023"):
        self.ARCHIVO = archivo
        self.YEARS_OBJETIVO = years_objetivo
        self.cargar_datos()
        self.configurar_metricas()

    def cargar_datos(self):
        """Carga y prepara los datos iniciales"""
        try:
            self.df = pd.read_csv(self.ARCHIVO)
            # Ordenar por RankingKAI ascendente (menor número = mejor posición)
            if "RankingKAI" in self.df.columns:
                self.df = self.df.sort_values("RankingKAI", ascending=True)
            print(f"✅ Datos cargados: {len(self.df)} instituciones")
            print(f"📊 Ordenadas por RankingKAI (1 = mejor posición)")
        except FileNotFoundError:
            print(f"❌ Archivo '{self.ARCHIVO}' no encontrado")
            self.df = pd.DataFrame()

    def configurar_metricas(self):
        """Configuración de métricas y pesos"""
        self.RENOMBRAMIENTO_METRICAS = {
            "excel_lider": "Excellence with Leadership",
            "normalized_impact": "Normalized Impact", "output": "Output en Scopus",
            "stp": "Scientific Talent Pool", "lider": "Scientific Leadership",
            "colab": "International Collaboration", "q1": "High Quality Publications (Q1)",
            "excel": "Excellence", "ik": "Innovative Knowledge", "tech_impact": "Technological Impact",
            "patents": "Patents", "open_access": "Open Access", "plumx": "Métricas PlumX",
            "mendeley": "Documentos en Mendeley", "overton": "Impact in public policy (Overton)",
            "sdg": "Sustainable Development Goals", "female_stp": "Female Scientific Talent Pool",
            "Altmetrics": "Altmetrics", "web_size": "Web Size", "AScore": "Authority Score",
            "impacto_score": "Puntaje calculado", "RankingKAI": "Ranking KAI"
        }

        self.METRICAS_ORIGINALES = {v: k for k, v in self.RENOMBRAMIENTO_METRICAS.items()}
        # Excluir IDs y columnas no numéricas
        self.COLUMNAS_EXCLUIDAS = ["Institucion", "impacto_score", "RankingKAI", "ID", "idp", "years", "lang"]

    def get_metricas_numericas(self, df):
        """Obtiene columnas numéricas para análisis"""
        return [col for col in df.columns
                if pd.api.types.is_numeric_dtype(df[col]) and col not in self.COLUMNAS_EXCLUIDAS]

    def mostrar_instituciones_disponibles(self, top_n=20):
        """Muestra instituciones disponibles ordenadas por ranking de menor a mayor (1 = mejor)"""
        if self.df.empty:
            print("❌ No hay datos cargados")
            return []

        print(f"\n🏆 INSTITUCIONES DISPONIBLES (Top {top_n} - Ranking de menor a mayor):")
        print("=" * 80)
        print(f"{'POS':<4} {'RANKING':<8} {'SCORE':<8} {'INSTITUCIÓN'}")
        print("-" * 80)

        instituciones = []
        # Tomar las primeras top_n instituciones (ya están ordenadas por RankingKAI ascendente)
        for i, (_, row) in enumerate(self.df.head(top_n).iterrows(), 1):
            nombre = row["Institucion"]
            ranking = row.get("RankingKAI", "N/A")
            score = row.get("impacto_score", 0)
            
            # Formatear la salida
            print(f"{i:<4} {ranking:<8} {score:<8.0f} {nombre}")
            instituciones.append((nombre, ranking, score))

        print("-" * 80)
        print(f"📌 NOTA: RankingKAI 1 = mejor posición, {self.df['RankingKAI'].max()} = última posición")
        
        return instituciones

    def mostrar_todas_instituciones(self):
        """Muestra todas las instituciones ordenadas por ranking"""
        if self.df.empty:
            print("❌ No hay datos cargados")
            return []

        print(f"\n📋 TODAS LAS INSTITUCIONES ({len(self.df)} total)")
        print("=" * 80)
        print(f"{'POS':<4} {'RANKING':<8} {'SCORE':<8} {'INSTITUCIÓN'}")
        print("-" * 80)

        instituciones = []
        for i, (_, row) in enumerate(self.df.iterrows(), 1):
            nombre = row["Institucion"]
            ranking = row.get("RankingKAI", "N/A")
            score = row.get("impacto_score", 0)
            
            print(f"{i:<4} {ranking:<8} {score:<8.0f} {nombre}")
            instituciones.append((nombre, ranking, score))

        return instituciones

    def seleccionar_institucion(self, mensaje="Selecciona institución", mostrar_todas=False):
        """Permite seleccionar una institución de la lista"""
        if mostrar_todas:
            instituciones_info = self.mostrar_todas_instituciones()
        else:
            instituciones_info = self.mostrar_instituciones_disponibles()

        if not instituciones_info:
            return None

        # Extraer solo los nombres para facilitar la selección
        instituciones_nombres = [info[0] for info in instituciones_info]

        while True:
            try:
                opcion = input(f"\n{mensaje} (1-{len(instituciones_nombres)} o nombre): ").strip()

                # Si es un número, seleccionar por índice
                if opcion.isdigit():
                    idx = int(opcion) - 1
                    if 0 <= idx < len(instituciones_nombres):
                        print(f"✅ Seleccionada: {instituciones_nombres[idx]}")
                        return instituciones_nombres[idx]
                    else:
                        print(f"❌ Opción debe estar entre 1 y {len(instituciones_nombres)}")
                # Si es texto, buscar por nombre
                else:
                    mask = self.df["Institucion"].str.contains(opcion, case=False, na=False)
                    if mask.any():
                        nombre_completo = self.df.loc[mask, "Institucion"].iloc[0]
                        ranking = self.df.loc[mask, "RankingKAI"].iloc[0]
                        print(f"✅ Encontrada: {nombre_completo} (Ranking #{ranking})")
                        return nombre_completo
                    else:
                        print(f"❌ No se encontró '{opcion}'")
                        print("💡 Prueba con: 'todas' para ver la lista completa")

            except (ValueError, KeyboardInterrupt):
                print("❌ Selección inválida")
                return None

    def calcular_score(self, data_row, metricas_numericas, variable_extra=None, valor_extra=None):
        """Calcula score basado en suma de métricas"""
        score = 0
        for col in metricas_numericas:
            valor = data_row.get(col, 0)
            if variable_extra and col == variable_extra:
                valor = valor_extra
            score += valor
        return score

    def analisis_brechas(self, data1, data2, metricas_numericas):
        """Identifica y prioriza brechas entre instituciones"""
        brechas = []
        for col in metricas_numericas:
            val1, val2 = data1.get(col, 0), data2.get(col, 0)
            if val2 > val1:  # Institución 2 es mejor
                brecha = val2 - val1
                impacto_potencial = brecha
                nombre = self.RENOMBRAMIENTO_METRICAS.get(col, col)
                brechas.append((nombre, brecha, impacto_potencial, col))

        # Ordenar por impacto y retornar top 5
        return sorted(brechas, key=lambda x: x[2], reverse=True)[:5]

    def simular_escenarios(self, data1, metricas_numericas, variable, porcentaje_mejora):
        """Simula diferentes escenarios de mejora"""
        valor_actual = data1[variable]
        nuevo_valor = valor_actual * (1 + porcentaje_mejora/100)
        score_mejorado = self.calcular_score(data1, metricas_numericas, variable, nuevo_valor)
        return nuevo_valor, score_mejorado

    def comparar_instituciones_avanzado(self, inst1, inst2, variable_simular=None, mejora_porcentaje=0):
        """Análisis avanzado 1vs1 con simulaciones"""
        if self.df.empty:
            print("❌ No hay datos cargados")
            return

        metricas_numericas = self.get_metricas_numericas(self.df)

        # Buscar instituciones
        mask1 = self.df["Institucion"].str.contains(inst1, case=False, na=False)
        mask2 = self.df["Institucion"].str.contains(inst2, case=False, na=False)

        if not mask1.any() or not mask2.any():
            print("❌ Una o ambas instituciones no encontradas")
            return

        data1, data2 = self.df.loc[mask1].iloc[0], self.df.loc[mask2].iloc[0]
        nombre_inst1, nombre_inst2 = data1["Institucion"], data2["Institucion"]

        # Análisis base
        score1 = self.calcular_score(data1, metricas_numericas)
        score2 = self.calcular_score(data2, metricas_numericas)

        print(f"\n🎯 ANÁLISIS AVANZADO: {nombre_inst1} vs {nombre_inst2}")
        print("=" * 70)

        # Resumen ejecutivo
        self.mostrar_resumen_ejecutivo(nombre_inst1, nombre_inst2, score1, score2, data1, data2)

        # Análisis detallado
        self.mostrar_analisis_detallado(data1, data2, metricas_numericas)

        # Análisis de brechas
        brechas = self.analisis_brechas(data1, data2, metricas_numericas)
        self.mostrar_analisis_brechas(brechas, nombre_inst2)

        # Simulación si se solicita
        if variable_simular:
            self.ejecutar_simulacion(data1, data2, metricas_numericas, variable_simular,
                                   mejora_porcentaje, nombre_inst1, nombre_inst2, score1, score2)

        return data1, data2, brechas

    def mostrar_resumen_ejecutivo(self, inst1, inst2, score1, score2, data1, data2):
        """Muestra resumen ejecutivo del análisis"""
        diferencia = score1 - score2
        ranking1 = data1.get("RankingKAI", "N/A")
        ranking2 = data2.get("RankingKAI", "N/A")

        print("📊 RESUMEN EJECUTIVO")
        print("-" * 60)
        print(f"🏆 INSTITUCIÓN LÍDER: {inst1 if diferencia > 0 else inst2}")
        print(f"📈 DIFERENCIA DE SCORE: {abs(diferencia):.0f} puntos")
        print(f"🏅 RANKINGS: #{ranking1} vs #{ranking2}")
        print(f"🎯 SCORES: {score1:.0f} | {score2:.0f}")
        print("-" * 60)

    def mostrar_analisis_detallado(self, data1, data2, metricas_numericas):
        """Muestra análisis detallado métrica por métrica"""
        print("\n🔍 ANÁLISIS DETALLADO POR MÉTRICA")
        print(f"{'MÉTRICA':<35} | {'INST1':>10} | {'INST2':>10} | {'DIF':>8} | {'ESTADO':>10}")
        print("-" * 80)

        for col in metricas_numericas:
            val1, val2 = data1.get(col, 0), data2.get(col, 0)
            if val1 == 0 and val2 == 0:
                continue

            dif = val1 - val2
            nombre = self.RENOMBRAMIENTO_METRICAS.get(col, col)[:34]

            if dif > 0:
                estado = "✅ VENTAJA"
            elif dif < 0:
                estado = "📉 DESVENTAJA"
            else:
                estado = "⚖️  IGUAL"

            print(f"{nombre:<35} | {val1:>10.0f} | {val2:>10.0f} | {dif:>7.0f} | {estado:>10}")

    def mostrar_analisis_brechas(self, brechas, inst_competidora):
        """Muestra análisis de brechas críticas"""
        print(f"\n📈 BRECHAS CRÍTICAS (vs {inst_competidora[:20]})")
        print("-" * 60)

        if not brechas:
            print("✅ No se identificaron brechas críticas")
            return

        for i, (nombre, brecha, impacto, col_original) in enumerate(brechas, 1):
            print(f"{i}. {nombre}:")
            print(f"   📏 Brecha: {brecha:.0f} puntos")
            print(f"   🎯 Impacto potencial: +{impacto:.0f} puntos en score")
            print(f"   💡 Prioridad: {'ALTA' if i <= 2 else 'MEDIA' if i <= 4 else 'BAJA'}")
            print()

    def ejecutar_simulacion(self, data1, data2, metricas_numericas, variable_simular,
                          mejora_porcentaje, nombre_inst1, nombre_inst2, score1, score2):
        """Ejecuta y muestra simulación de mejora"""
        nombre_original = self.METRICAS_ORIGINALES.get(variable_simular)

        if nombre_original and nombre_original in metricas_numericas:
            print(f"\n🔄 SIMULACIÓN ESTRATÉGICA")
            print("-" * 50)

            nuevo_valor, score_mejorado = self.simular_escenarios(
                data1, metricas_numericas, nombre_original, mejora_porcentaje
            )

            ganancia = score_mejorado - score1

            print(f"📊 MÉTRICA: {variable_simular}")
            print(f"🔄 CAMBIO: {data1[nombre_original]:.0f} → {nuevo_valor:.0f} (+{mejora_porcentaje}%)")
            print(f"📈 SCORE: {score1:.0f} → {score_mejorado:.0f} (+{ganancia:.0f})")

            # Análisis de impacto
            if score_mejorado > score2 and score1 <= score2:
                print("🎉 IMPACTO: ¡CAMBIO DE LIDERAZGO!")
                print(f"   {nombre_inst1[:20]} ahora supera a {nombre_inst2[:20]}")
            elif score_mejorado > score2:
                print("📈 IMPACTO: Ampliación de ventaja competitiva")
            else:
                deficit = score2 - score_mejorado
                print(f"⚠️  IMPACTO: Se reduce brecha, pero aún falta {deficit:.0f} puntos")

    def analisis_comparativo_top10(self, institucion):
        """Análisis comparativo vs promedio del Top 10"""
        if self.df.empty:
            print("❌ No hay datos cargados")
            return

        metricas_numericas = self.get_metricas_numericas(self.df)

        # Calcular Top 10 (menor RankingKAI = mejor)
        if "RankingKAI" not in self.df.columns:
            print("❌ Se requiere columna 'RankingKAI' para este análisis")
            return

        top10 = self.df.nsmallest(10, "RankingKAI")
        promedio_top10 = top10[metricas_numericas].mean()

        # Buscar institución
        mask = self.df["Institucion"].str.contains(institucion, case=False, na=False)
        if not mask.any():
            print(f"❌ Institución '{institucion}' no encontrada")
            return

        data_inst = self.df.loc[mask].iloc[0]
        nombre_inst = data_inst["Institucion"]
        ranking_inst = data_inst.get("RankingKAI", "N/A")

        print(f"\n🏆 ANÁLISIS VS TOP 10: {nombre_inst} (Ranking #{ranking_inst})")
        print("=" * 70)

        score_inst = self.calcular_score(data_inst, metricas_numericas)
        score_promedio = self.calcular_score(promedio_top10, metricas_numericas)

        print(f"📊 SCORE INSTITUCIÓN: {score_inst:.0f}")
        print(f"📊 SCORE PROMEDIO TOP 10: {score_promedio:.0f}")
        print(f"📈 DIFERENCIA: {score_inst - score_promedio:+.0f} puntos")

        if score_inst > score_promedio:
            print("✅ POR ENCIMA del promedio del Top 10")
        else:
            print("📉 POR DEBAJO del promedio del Top 10")

        # Mostrar ranking dentro del Top 10 si aplica
        if ranking_inst != "N/A" and ranking_inst <= 10:
            posicion = ranking_inst
            print(f"🏅 POSICIÓN EN TOP 10: #{posicion}")

    def seleccionar_variable_simulacion(self, metricas_numericas):
        """Permite seleccionar variable para simulación"""
        variables_display = [self.RENOMBRAMIENTO_METRICAS.get(k, k) for k in metricas_numericas]

        print("\n📊 VARIABLES DISPONIBLES PARA SIMULACIÓN:")
        for i, var in enumerate(variables_display, 1):
            print(f"   {i}. {var}")

        while True:
            try:
                opcion = input(f"\nSelecciona variable (1-{len(variables_display)}): ").strip()
                if opcion.isdigit():
                    idx = int(opcion) - 1
                    if 0 <= idx < len(variables_display):
                        return variables_display[idx]
                print(f"❌ Opción debe estar entre 1 y {len(variables_display)}")
            except (ValueError, KeyboardInterrupt):
                return None

    def modo_interactivo(self):
        """Modo interactivo para análisis continuo"""
        if self.df.empty:
            print("❌ No se pueden cargar los datos")
            return

        print(f"\n🔧 HERRAMIENTA DE ANÁLISIS AVANZADO ({self.YEARS_OBJETIVO})")
        print("=" * 80)
        print("📊 LAS INSTITUCIONES SE MUESTRAN ORDENADAS POR RANKINGKAI (1 = MEJOR POSICIÓN)")
        print("=" * 80)

        while True:
            print("\n🎯 OPCIONES DE ANÁLISIS:")
            print("1. Comparación 1vs1 avanzada")
            print("2. Análisis vs Top 10")
            print("3. Ver lista completa de instituciones")
            print("4. Salir")

            opcion = input("\nSelecciona opción (1-4): ").strip()

            if opcion == "4":
                break
            elif opcion == "1":
                self.ejecutar_comparacion_1vs1()
            elif opcion == "2":
                self.ejecutar_analisis_top10()
            elif opcion == "3":
                self.mostrar_todas_instituciones()
                input("\nPresiona Enter para continuar...")
            else:
                print("❌ Opción inválida")

    def ejecutar_comparacion_1vs1(self):
        """Ejecuta comparación 1vs1 en modo interactivo"""
        print("\n🏫 COMPARACIÓN 1vs1 AVANZADA")

        # Selección de instituciones
        print("\n--- SELECCIÓN INSTITUCIÓN 1 ---")
        inst1 = self.seleccionar_institucion("Selecciona Institución 1")
        if not inst1:
            return

        print("\n--- SELECCIÓN INSTITUCIÓN 2 ---")
        inst2 = self.seleccionar_institucion("Selecciona Institución 2")
        if not inst2:
            return

        if inst1 == inst2:
            print("❌ Debes seleccionar instituciones diferentes")
            return

        # Simulación
        metricas_numericas = self.get_metricas_numericas(self.df)
        variable, porcentaje = None, 0

        simular = input("\n¿Incluir simulación? (s/n): ").strip().lower()
        if simular == 's':
            variable = self.seleccionar_variable_simulacion(metricas_numericas)
            if variable:
                try:
                    porcentaje = float(input("Porcentaje de mejora: "))
                except ValueError:
                    print("❌ Porcentaje inválido")

        # Ejecutar análisis
        self.comparar_instituciones_avanzado(inst1, inst2, variable, porcentaje)

    def ejecutar_analisis_top10(self):
        """Ejecuta análisis vs Top 10 en modo interactivo"""
        institucion = self.seleccionar_institucion("Selecciona institución para análisis vs Top 10")
        if institucion:
            self.analisis_comparativo_top10(institucion)

# Ejecución principal
if __name__ == "__main__":
    analizador = AnalizadorInstitucionesAvanzado()
    analizador.modo_interactivo()

✅ Datos cargados: 57 instituciones
📊 Ordenadas por RankingKAI (1 = mejor posición)

🔧 HERRAMIENTA DE ANÁLISIS AVANZADO (2019-2023)
📊 LAS INSTITUCIONES SE MUESTRAN ORDENADAS POR RANKINGKAI (1 = MEJOR POSICIÓN)

🎯 OPCIONES DE ANÁLISIS:
1. Comparación 1vs1 avanzada
2. Análisis vs Top 10
3. Ver lista completa de instituciones
4. Salir
❌ Opción inválida

🎯 OPCIONES DE ANÁLISIS:
1. Comparación 1vs1 avanzada
2. Análisis vs Top 10
3. Ver lista completa de instituciones
4. Salir

📋 TODAS LAS INSTITUCIONES (57 total)
POS  RANKING  SCORE    INSTITUCIÓN
--------------------------------------------------------------------------------
1    1        105896   Universidad de Chile
2    2        96899    Pontificia Universidad Catolica de Chile
3    3        49647    Universidad de Concepcion
4    4        32984    Universidad Andres Bello, Chile
5    5        30832    Universidad de Santiago de Chile
6    6        28100    Universidad Austral de Chile
7    7        27017    Pontificia Universidad Catol